# Experiment 1 — UCI Default of Credit Card Clients

Reproduces:
- **Table 1**: MSE / AUC / KS with 95% BCa CIs (2000 bootstrap resamples)
  - Scorecard: AUC=0.7599, KS=0.4018, MSE=0.1389
  - GBM: AUC=0.7727, KS=0.4203, MSE=0.1359
  - QPM (latent): AUC=0.7773, KS=0.4289, MSE=0.1359
  - QPM (quantized): AUC=0.7750, KS=0.4225, MSE=0.1359
- **Table 2**: 14-bin QPM score ladder (monotone PD centers)
- **Figure 1**: GBM continuous vs QPM step function
- **Section 3.4**: Churn scenarios A / B / C + GBM head-to-head

**Dataset**: UCI Default of Credit Card Clients (direct CSV download — no extra packages)
30,000 accounts · 22.1% default rate · 25 features

**First run**: the notebook downloads the CSV, prints its SHA-256, and saves split indices.
Set `EXPECTED_DATA_SHA256` in the config cell to lock the dataset version for future runs.

**Installation** (run once from repo root):
```bash
pip install -e ..
```

In [ ]:
# ── CONFIGURATION — edit here, then Kernel → Restart & Run All ────────────────
# [1] Single-entry config block
FAST_RUN = False      # True → <3 min smoke-test on CPU
DEVICE   = "cpu"      # xgboost tree_method hint

# [6] All seeds declared explicitly
SEED_SPLIT = 42
SEED_GBM   = 857
SEED_BOOT  = 0

# Paths
import pathlib
ARTIFACTS  = pathlib.Path("../artifacts/exp01")
SPLITS_DIR = ARTIFACTS / "splits"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(exist_ok=True)

# Hyper-parameters — [8] FAST_RUN overrides for speed
N_ESTIMATORS     = 100  if FAST_RUN else 400
N_BOOT           = 200  if FAST_RUN else 2000
LEARNING_RATE    = 0.03
MAX_DEPTH        = 4
SUBSAMPLE        = 0.7
COLSAMPLE_BYTREE = 0.7
MIN_CHILD_WEIGHT = 10
GAMMA            = 2

# [3] Dataset URL — swap '/refs/heads/main' for a commit SHA to pin exactly
DATASET_URL_PINNED = (
    "https://raw.githubusercontent.com/MatteoM95/"
    "Default-of-Credit-Card-Clients-Dataset-Analisys/"
    "refs/heads/main/dataset/default_of_credit_card_clients.csv"
)
# Paste the SHA-256 printed on first run here to enforce dataset immutability
EXPECTED_DATA_SHA256 = ""

print(f"FAST_RUN={FAST_RUN}  |  N_ESTIMATORS={N_ESTIMATORS}  |  N_BOOT={N_BOOT}")
print(f"SEED_SPLIT={SEED_SPLIT}  |  SEED_GBM={SEED_GBM}  |  SEED_BOOT={SEED_BOOT}")
if not EXPECTED_DATA_SHA256:
    print("NOTE: set EXPECTED_DATA_SHA256 after first run to lock dataset version.")

In [ ]:
import os, sys, random, hashlib, json as _json, pathlib, importlib, subprocess, zipfile
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import xgboost as xgb

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from qpm import (
    QPMQuantizer, AnchoredCalibration,
    auc_score, ks_score, mse_score, bootstrap_ci, delong_test, churn_metrics,
    WoEScorecard,
)


# [2] Determinism helper — call before every model fit
def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)


# [5] Environment lock — print all package versions
_PKGS = ["numpy", "pandas", "sklearn", "xgboost", "scipy", "matplotlib"]
print(f"Python {sys.version.split()[0]}")
for _p in _PKGS:
    try:
        _mod = importlib.import_module(_p)
        print(f"  {_p}: {_mod.__version__}")
    except Exception:
        print(f"  {_p}: NOT FOUND")

In [ ]:
# [2] Determinism probe — runs before any modelling; must pass
seed_everything(SEED_GBM)
_Xp = np.random.default_rng(0).random((200, 8))
_yp = (_Xp[:, 0] + 0.3 * _Xp[:, 1] > 0.7).astype(float)

def _quick_gbm(s):
    # subsample + colsample_bytree ensure the random_state actually drives
    # different tree structures, making the same/different-seed check reliable
    m = xgb.XGBRegressor(n_estimators=20, learning_rate=0.1,
                          subsample=0.7, colsample_bytree=0.7,
                          random_state=s, tree_method="hist")
    m.fit(_Xp, _yp)
    return m.predict(_Xp[:5])

_p1, _p2, _p3 = _quick_gbm(42), _quick_gbm(42), _quick_gbm(99)
assert np.allclose(_p1, _p2), "FAIL: same seed produced different predictions!"
assert not np.allclose(_p1, _p3), "FAIL: different seeds produced identical predictions!"
print("Determinism probe PASSED")
del _Xp, _yp, _p1, _p2, _p3

In [ ]:
# [3] Data-provenance hash utilities
def _sha256_df(df, nrows=None):
    sub = df if nrows is None else df.iloc[:nrows]
    return hashlib.sha256(sub.to_csv(index=False).encode()).hexdigest()


def _verify_hash(name, h, cache_path):
    cache = pathlib.Path(cache_path)
    known = _json.loads(cache.read_text()) if cache.exists() else {}
    if name in known:
        assert known[name] == h, (
            f"Data hash mismatch for '{name}'! "
            f"Expected {known[name][:16]}... got {h[:16]}... "
            "Delete artifacts/*/data_hashes.json to reset."
        )
        print(f"  {name}: hash OK")
    else:
        known[name] = h
        cache.write_text(_json.dumps(known, indent=2))
        print(f"  {name}: hash saved (first run) -- {h[:16]}...")


# [4] Deterministic splits — indices persisted to disk
def _get_or_create_splits(X, y, test_size, seed, tag, splits_dir):
    sd = pathlib.Path(splits_dir)
    tr_p = sd / f"{tag}_train.npy"
    te_p = sd / f"{tag}_test.npy"
    if tr_p.exists() and te_p.exists():
        tr_idx, te_idx = np.load(str(tr_p)), np.load(str(te_p))
        print(f"  Loaded split '{tag}' from disk")
    else:
        all_idx = np.arange(len(y))
        tr_idx, te_idx = train_test_split(
            all_idx, test_size=test_size, random_state=seed, stratify=y
        )
        np.save(str(tr_p), tr_idx)
        np.save(str(te_p), te_idx)
        print(f"  Created + saved split '{tag}'")
    return tr_idx, te_idx

## 1. Download & Load Data

In [ ]:
# Source: UCI Default of Credit Card Clients
# https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
import urllib.request, io

if "/refs/heads/" in DATASET_URL_PINNED:
    print("WARNING: URL is branch-based. Replace with a commit-pinned URL for strict reproducibility.")

print(f"Downloading: {DATASET_URL_PINNED.split('/')[-1]}")
with urllib.request.urlopen(DATASET_URL_PINNED, timeout=60) as resp:
    csv_bytes = resp.read()

# [3] Data provenance: hash the raw bytes before any parsing
data_sha256 = hashlib.sha256(csv_bytes).hexdigest()
if EXPECTED_DATA_SHA256:
    if data_sha256 != EXPECTED_DATA_SHA256:
        raise ValueError(
            f"Dataset hash mismatch!\n"
            f"  Expected: {EXPECTED_DATA_SHA256}\n"
            f"  Got:      {data_sha256}\n"
            "Update EXPECTED_DATA_SHA256 in the config cell if this is intentional."
        )
    print(f"  Hash verified: {data_sha256[:16]}...")
else:
    print(f"  SHA-256: {data_sha256}")
    print("  Set EXPECTED_DATA_SHA256 = above value in config to lock dataset version.")

df = pd.read_csv(io.BytesIO(csv_bytes))

# Identify the target column (handles header row variation across CSV sources)
_target_candidates = ["default payment next month", "default.payment.next.month", "Y"]
target_col = next((c for c in _target_candidates if c in df.columns), None)
if target_col is None:
    target_col = df.columns[-1]
    print(f"  Auto-detected target column: '{target_col}'")
else:
    print(f"  Target column: '{target_col}'")

y_all = df[target_col].values.astype(float)
X_df  = df.drop(columns=[target_col])
# Drop the ID column if present
for _id_col in ["ID", "id"]:
    if _id_col in X_df.columns:
        X_df = X_df.drop(columns=[_id_col])
X_all = X_df.values.astype(float)

print(f"Samples: {len(y_all):,}  |  Features: {X_all.shape[1]}  |  DR: {y_all.mean():.1%}")
print("  Paper: 30,000 accounts, 22.1% default rate")

## 2. Train / Test Split

In [ ]:
# [4] Split indices saved to disk — guaranteed identical across runs
print("Loading/creating train-test split...")
train_idx, test_idx = _get_or_create_splits(
    X_all, y_all, test_size=6000, seed=SEED_SPLIT, tag="main", splits_dir=SPLITS_DIR
)
X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print(f"Train DR: {y_train.mean():.1%}  |  Test DR: {y_test.mean():.1%}")

## 3. Scorecard Baseline

WoE logistic regression — 10 equal-frequency quantile bins per feature.

In [ ]:
sc = WoEScorecard()
sc.fit(X_train, y_train)
F_sc_test = sc.predict_proba(X_test)

sc_auc = auc_score(y_test, F_sc_test)
sc_ks  = ks_score(y_test, F_sc_test)
sc_mse = mse_score(y_test, F_sc_test)

print(f"Scorecard  AUC: {sc_auc:.4f}  KS: {sc_ks:.4f}  MSE: {sc_mse:.4f}")
print(f"  Paper:   AUC: 0.7599      KS: 0.4018      MSE: 0.1389")

## 4. GBM Baseline

Out-of-the-box XGBoost — no hyperparameter tuning, consistent with the paper's protocol.

In [ ]:
# [6] Explicit random_state in every model call
seed_everything(SEED_GBM)
gbm = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,      # [8] FAST_RUN reduces this
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=GAMMA,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED_GBM,          # [6] explicit
)
gbm.fit(X_train, y_train)
F_train_gbm = gbm.predict(X_train)
F_test_gbm  = gbm.predict(X_test)

gbm_auc = auc_score(y_test, F_test_gbm)
gbm_ks  = ks_score(y_test, F_test_gbm)
gbm_mse = mse_score(y_test, F_test_gbm)

print(f"GBM       AUC: {gbm_auc:.4f}  KS: {gbm_ks:.4f}  MSE: {gbm_mse:.4f}")
print(f"  Paper:  AUC: 0.7727      KS: 0.4203      MSE: 0.1359")

## 5. QPM on top of GBM

Non-uniform supervised binning with concentrated resolution in the focus region.

In [ ]:
qpm = QPMQuantizer(
    focus_pd=(0.05, 0.20),
    n_low_max=6, n_mid_max=6, n_high_max=6,
    min_band_size=300,
    pd_cap=0.80,
    monotone_smooth=True,
)
qpm.fit(F_train_gbm, y_train)

F_lat = F_test_gbm               # latent (= GBM output, unchanged)
F_qpm = qpm.predict(F_test_gbm)  # quantized PD ladder

qpm_lat_auc = auc_score(y_test, F_lat)
qpm_lat_ks  = ks_score(y_test, F_lat)
qpm_lat_mse = mse_score(y_test, F_lat)
qpm_q_auc   = auc_score(y_test, F_qpm)
qpm_q_ks    = ks_score(y_test, F_qpm)
qpm_q_mse   = mse_score(y_test, F_qpm)

print(f"K={qpm.n_bins_} bins  (paper: 14)  "
      f"n_low={qpm.chosen_n_[0]}, n_mid={qpm.chosen_n_[1]}, n_high={qpm.chosen_n_[2]}")
print(f"QPM latent    AUC: {qpm_lat_auc:.4f}  KS: {qpm_lat_ks:.4f}  MSE: {qpm_lat_mse:.4f}")
print(f"  Paper:      AUC: 0.7773      KS: 0.4289      MSE: 0.1359")
print(f"QPM quantized AUC: {qpm_q_auc:.4f}  KS: {qpm_q_ks:.4f}  MSE: {qpm_q_mse:.4f}")
print(f"  Paper:      AUC: 0.7750      KS: 0.4225      MSE: 0.1359")

## 6. Table 1 — AUC / KS / MSE with 95% BCa CIs

2000 bootstrap resamples with patient-level sampling (Appendix E).

In [ ]:
print(f"Computing BCa CIs ({N_BOOT} resamples, seed={SEED_BOOT})...")


def _bca(y, f, fn):
    return bootstrap_ci(y, f, fn, n_boot=N_BOOT, method="bca", seed=SEED_BOOT)


sc_auc_lo,  sc_auc_hi  = _bca(y_test, F_sc_test,   auc_score)
sc_ks_lo,   sc_ks_hi   = _bca(y_test, F_sc_test,   ks_score)
gbm_auc_lo, gbm_auc_hi = _bca(y_test, F_test_gbm,  auc_score)
gbm_ks_lo,  gbm_ks_hi  = _bca(y_test, F_test_gbm,  ks_score)
ql_auc_lo,  ql_auc_hi  = _bca(y_test, F_lat,        auc_score)
ql_ks_lo,   ql_ks_hi   = _bca(y_test, F_lat,        ks_score)
qq_auc_lo,  qq_auc_hi  = _bca(y_test, F_qpm,        auc_score)
qq_ks_lo,   qq_ks_hi   = _bca(y_test, F_qpm,        ks_score)

print(f"\n{'Model':<19} {'MSE':>7} {'AUC':>7} {'95% CI AUC':>17} {'KS':>7} {'95% CI KS':>17}")
print("-" * 78)
_rows = [
    ("Scorecard",       sc_mse,      sc_auc,      sc_auc_lo,  sc_auc_hi,  sc_ks,      sc_ks_lo,  sc_ks_hi),
    ("GBM",             gbm_mse,     gbm_auc,     gbm_auc_lo, gbm_auc_hi, gbm_ks,     gbm_ks_lo, gbm_ks_hi),
    ("QPM (latent)",    qpm_lat_mse, qpm_lat_auc, ql_auc_lo,  ql_auc_hi,  qpm_lat_ks, ql_ks_lo,  ql_ks_hi),
    ("QPM (quantized)", qpm_q_mse,   qpm_q_auc,   qq_auc_lo,  qq_auc_hi,  qpm_q_ks,   qq_ks_lo,  qq_ks_hi),
]
for name, mse, auc, alo, ahi, ks, klo, khi in _rows:
    print(f"{name:<19} {mse:.4f}  {auc:.4f} [{alo:.3f}-{ahi:.3f}]  "
          f"{ks:.4f} [{klo:.3f}-{khi:.3f}]")

## 7. Table 2 — QPM Score Ladder

In [ ]:
ladder = qpm.score_ladder(F_test_gbm, y_test)
print(f"QPM score ladder — {len(ladder)} strata  (paper: 14 bins)")
print(f"PD range: {qpm.bin_reps_.min():.1%} to {qpm.bin_reps_.max():.1%}")
ladder

## 8. Figure 1 — GBM continuous vs QPM step function

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sort_idx = np.argsort(F_test_gbm)

ax.scatter(range(len(y_test)), y_test[sort_idx], s=2, c="grey", alpha=0.3,
           label="Observed default")
ax.plot(range(len(y_test)), F_test_gbm[sort_idx], lw=0.8, c="steelblue",
        label="GBM (continuous)")
ax.step(range(len(y_test)), F_qpm[sort_idx], lw=2, c="darkorange",
        label="QPM (ladder)")

ax.set_xlabel("Accounts (sorted by GBM score)")
ax.set_ylabel("Predicted PD")
ax.set_title("Figure 1 — UCI Default: GBM continuous vs QPM risk ladder")
ax.legend(fontsize=9)
fig.tight_layout()

fig_path = ARTIFACTS / "figure1_uci_risk_ladder.png"
fig.savefig(str(fig_path), dpi=150)
plt.show()
print(f"Saved: {fig_path}")

## 9. Churn Analysis (Section 3.4)

Compares three update scenarios:
- **Scenario A** — Independent QPM retrain (new edges each time)
- **Scenario B** — Fixed edges, new GBM latent scores (fine-tune)
- **Scenario C** — AnchoredCalibration on same F: guaranteed 0% churn (Proposition 1)

In [ ]:
# Slice training data into two halves (simulates periodic retraining)
N_HALF = len(y_train) // 2
X_s1, y_s1 = X_train[:N_HALF], y_train[:N_HALF]
X_s2, y_s2 = X_train[N_HALF:2 * N_HALF], y_train[N_HALF:2 * N_HALF]
print(f"Slice1: {len(y_s1):,}  |  Slice2: {len(y_s2):,}")


def _make_gbm():
    return xgb.XGBRegressor(
        n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH, subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE_BYTREE, min_child_weight=MIN_CHILD_WEIGHT,
        gamma=GAMMA, objective="reg:squarederror",
        tree_method="hist", random_state=SEED_GBM,
    )


# GBM v1 trained on slice1, GBM v2 retrained on slice2
seed_everything(SEED_GBM)
gbm_v1 = _make_gbm(); gbm_v1.fit(X_s1, y_s1)
seed_everything(SEED_GBM)
gbm_v2 = _make_gbm(); gbm_v2.fit(X_s2, y_s2)

F_s1_tr = gbm_v1.predict(X_s1)
F_v1    = gbm_v1.predict(X_test)   # v1 GBM scores on test set
F_v2    = gbm_v2.predict(X_test)   # v2 GBM scores on test set

# Global band churn: GBM retrain (5 percentile bands)
gbm_ch = churn_metrics(F_v1, F_v2, n_global_bands=5)
print(f"GBM retrain  global band churn: {gbm_ch['band_churn_rate']:.4f}")

# QPM v1 fitted on slice1 GBM scores
qpm_v1 = QPMQuantizer(focus_pd=(0.05, 0.20), n_low_max=6, n_mid_max=6, n_high_max=6,
                       min_band_size=300, pd_cap=0.80)
qpm_v1.fit(F_s1_tr, y_s1)
bins_v1 = qpm_v1.get_bin_index(F_v1)

# Scenario A: independent QPM retrain (new GBM, new edges)
F_s2_tr = gbm_v2.predict(X_s2)
qpm_v2a = QPMQuantizer(focus_pd=(0.05, 0.20), n_low_max=6, n_mid_max=6, n_high_max=6,
                        min_band_size=300, pd_cap=0.80)
qpm_v2a.fit(F_s2_tr, y_s2)
bins_v2a = qpm_v2a.get_bin_index(F_v2)
churn_a  = churn_metrics(F_v1, F_v2, bins_v1, bins_v2a, n_global_bands=5)
print(f"Scenario A (independent retrain): native churn = {churn_a['native_band_churn_rate']:.4f}")

# Scenario B: fixed v1 edges, new GBM v2 scores
bins_v2b = qpm_v1.get_bin_index(F_v2)
churn_b  = churn_metrics(F_v1, F_v2, bins_v1, bins_v2b, n_global_bands=5)
print(f"Scenario B (fixed edges):         native churn = {churn_b['native_band_churn_rate']:.4f}")

factor_ba = (churn_a["native_band_churn_rate"] /
             max(churn_b["native_band_churn_rate"], 1e-9))
print(f"  Reduction factor B vs A: {factor_ba:.1f}x  (paper: >4x)")

# Scenario C: same F, same edges => zero churn by Proposition 1
bins_v2c = qpm_v1.get_bin_index(F_v1)    # identical to bins_v1
churn_c  = churn_metrics(F_v1, F_v1, bins_v1, bins_v2c, n_global_bands=5)
print(f"Scenario C (AnchoredCalibration): native churn = {churn_c['native_band_churn_rate']:.4f}")
print(f"  Paper: 0.00% rank-order churn (Proposition 1)")
assert churn_c["native_band_churn_rate"] == 0.0, "Scenario C must be zero churn!"

In [ ]:
# [7] Results contract — collect all metrics
report = {
    "experiment": "01_uci_default",
    "fast_run": FAST_RUN,
    "seeds": {"split": SEED_SPLIT, "gbm": SEED_GBM, "boot": SEED_BOOT},
    "n_estimators": N_ESTIMATORS,
    "n_boot": N_BOOT,
    "scorecard": {"auc": sc_auc, "ks": sc_ks, "mse": sc_mse},
    "gbm": {"auc": gbm_auc, "ks": gbm_ks, "mse": gbm_mse},
    "qpm": {
        "n_bins": qpm.n_bins_,
        "bin_reps": qpm.bin_reps_.tolist(),
        "latent":    {"auc": qpm_lat_auc, "ks": qpm_lat_ks, "mse": qpm_lat_mse},
        "quantized": {"auc": qpm_q_auc,   "ks": qpm_q_ks,   "mse": qpm_q_mse},
    },
    "churn": {
        "scenario_a_native": churn_a["native_band_churn_rate"],
        "scenario_b_native": churn_b["native_band_churn_rate"],
        "scenario_c_native": churn_c["native_band_churn_rate"],
        "reduction_factor_b_vs_a": factor_ba,
    },
}

report_path = ARTIFACTS / "report.json"
report_path.write_text(_json.dumps(report, indent=2))
print(f"Report saved: {report_path}")

In [ ]:
# [7] Final assertions — paper claims verified programmatically
assert report["gbm"]["auc"] > 0.75, f"GBM AUC below threshold: {report['gbm']['auc']:.4f}"
assert report["qpm"]["n_bins"] >= 10, f"Expected >=10 bins, got {report['qpm']['n_bins']}"
assert report["churn"]["scenario_c_native"] == 0.0, "Scenario C must be zero churn!"

reps = np.array(report["qpm"]["bin_reps"])
assert (np.diff(reps) >= -1e-9).all(), "QPM bin reps are not monotone!"

print("=" * 45)
print("All assertions PASSED")
print(f"  GBM AUC:    {report['gbm']['auc']:.4f}  (paper: 0.7727)")
print(f"  QPM bins:   {report['qpm']['n_bins']}        (paper: 14)")
print(f"  Churn C:    {report['churn']['scenario_c_native']:.4f}  (paper: 0.00%)")
print(f"  Factor B/A: {report['churn']['reduction_factor_b_vs_a']:.1f}x  (paper: >4x)")